# Orthogonal Persona Vectors

Goal of this notebook: **find orthogonal persona vectors**, and **find prompts that
activate more than one persona at once**.

It is distilled from the ARENA `[4.4] LLM Psychology & Persona Vectors` exercises
(sections 3-4), built on **Qwen2.5-7B-Instruct** so it fits on a single 16GB GPU.
The heavyweight Gemma-2-27B / Qwen3-32B work from sections 1-2 is intentionally left out.

Workflow:
1. Load Qwen2.5-7B + the precomputed contrastive trait vectors (`sycophantic`, `evil`, `hallucinating`).
2. Measure the geometry of those vectors (cosine similarity ⇒ orthogonality).
3. Project candidate prompts onto every trait vector to find prompts that activate multiple personas.
4. (Optional) Extract brand-new persona vectors via contrastive prompting, to grow the set.


## Setup

Standalone repo (decoupled from ARENA). The cell below does the imports and points
`section_dir` at a local `data/` directory where every cache lives — the persona-pool
vectors/responses, the boundary maps, etc.

In [ ]:
import gc
import json
import os
import re
import sys
import textwrap
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import torch as t
import torch.nn.functional as F
from jaxtyping import Float
from torch import Tensor
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedModel, PreTrainedTokenizerBase

t.set_grad_enabled(False)
warnings.filterwarnings("ignore")

# --- Local data directory (standalone; no ARENA dependency) ---
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
section_dir = DATA_DIR          # kept name so downstream cache paths are unchanged
sys.path.insert(0, str(Path.cwd()))   # so the local `patchscope` module imports (Section 7)

device = t.device("cuda" if t.cuda.is_available() else "cpu")
dtype = t.bfloat16
MAIN = __name__ == "__main__"

print(f"data_dir = {section_dir.resolve()}")
print(f"device   = {device}")


def print_with_wrap(s: str, width: int = 80):
    """Print text with line wrapping, preserving newlines."""
    out = []
    for line in s.splitlines(keepends=False):
        out.append(textwrap.fill(line, width=width) if line.strip() else line)
    print("\n".join(out))

In [ ]:
# Type-hint imports used by the function annotations throughout the notebook.
# Kept separate from the (heavy) setup cell above so you can run just this without
# reloading the model. Safe to re-run anytime.
from jaxtyping import Float
from torch import Tensor
from transformers import PreTrainedModel, PreTrainedTokenizerBase

## Load Qwen2.5-7B-Instruct

~15GB in bf16 — fits a 16GB card with room for activations. Generation is sequential, so
keep `max_new_tokens` modest in experiments below.

In [ ]:
QWEN_SMALL_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

print(f"Loading {QWEN_SMALL_MODEL_NAME}...")
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_SMALL_MODEL_NAME)
qwen_model_small = AutoModelForCausalLM.from_pretrained(
    QWEN_SMALL_MODEL_NAME,
    dtype=dtype,
).to(device)

QWEN_NUM_LAYERS = qwen_model_small.config.num_hidden_layers
QWEN_D_MODEL = qwen_model_small.config.hidden_size
print(f"Model loaded with {QWEN_NUM_LAYERS} layers, hidden size {QWEN_D_MODEL}")

# Trait vectors are saved per-layer. Index conventions (from the ARENA exercises):
#   trait_vectors[L]            corresponds to the output of model.layers[L]
#   == hidden_states[L + 1]     (hidden_states[0] is the embedding layer)
TRAIT_VECTOR_LAYER = 20  # hidden_states index used when extracting activations
STEER_LAYER = TRAIT_VECTOR_LAYER - 1  # 0-based model.layers index; also indexes trait_vectors[L]

## Helper functions

`format_messages` / `extract_response_activations` give us mean residual-stream activations
over the assistant's response tokens; `extract_all_layer_activations_qwen` does the same at
every layer (needed if you re-extract vectors). `_return_layers` locates the transformer
blocks for hooking.

In [ ]:
def _normalize_messages(messages: list[dict[str, str]]) -> list[dict[str, str]]:
    """Merge any leading system message into the first user message (harmless for Qwen)."""
    if not messages or messages[0]["role"] != "system":
        return messages
    system_content = messages[0]["content"]
    rest = list(messages[1:])
    if rest and rest[0]["role"] == "user" and system_content:
        rest[0] = {"role": "user", "content": f"{system_content}\n\n{rest[0]['content']}"}
    return rest


def format_messages(messages: list[dict[str, str]], tokenizer: PreTrainedTokenizerBase) -> tuple[str, int]:
    """Format a conversation with the chat template; return (full_prompt, response_start_idx)."""
    messages = _normalize_messages(messages)
    full_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    prompt_without_response = tokenizer.apply_chat_template(
        messages[:-1], tokenize=False, add_generation_prompt=True
    ).rstrip()
    response_start_idx = tokenizer(prompt_without_response, return_tensors="pt").input_ids.shape[1] + 1
    return full_prompt, response_start_idx


def _return_layers(m: PreTrainedModel) -> list:
    """Locate the list of transformer blocks across architectures."""
    for attr_path in ("language_model.layers", "layers"):
        obj = m.model
        try:
            for name in attr_path.split("."):
                obj = getattr(obj, name)
            return obj
        except AttributeError:
            continue
    raise AttributeError(f"Could not find transformer layers on {type(m)}")


def extract_response_activations(
    model: PreTrainedModel, tokenizer: PreTrainedTokenizerBase,
    system_prompts: list[str], questions: list[str], responses: list[str],
    layer: int,
) -> Float[Tensor, "num_examples d_model"]:
    """Mean activation over response tokens at a single `hidden_states` layer index."""
    assert len(system_prompts) == len(questions) == len(responses)
    all_mean_activations = []
    for system_prompt, question, response in zip(system_prompts, questions, responses):
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
            {"role": "assistant", "content": response},
        ]
        full_prompt, response_start_idx = format_messages(messages, tokenizer)
        tokens = tokenizer(full_prompt, return_tensors="pt").to(model.device)
        with t.inference_mode():
            outputs = model(**tokens, output_hidden_states=True)
        hidden_states = outputs.hidden_states[layer]  # (1, seq_len, d_model)
        seq_len = hidden_states.shape[1]
        response_mask = t.arange(seq_len, device=hidden_states.device) >= response_start_idx
        mean_activation = (hidden_states[0] * response_mask[:, None]).sum(0) / response_mask.sum()
        all_mean_activations.append(mean_activation.cpu())
        del outputs
        t.cuda.empty_cache()
    return t.stack(all_mean_activations)


def extract_all_layer_activations_qwen(
    model: PreTrainedModel, tokenizer: PreTrainedTokenizerBase,
    system_prompts: list[str], questions: list[str], responses: list[str],
) -> Float[Tensor, "num_examples num_layers d_model"]:
    """Mean activation over response tokens at ALL layers (for re-extracting trait vectors)."""
    assert len(system_prompts) == len(questions) == len(responses)
    num_layers = model.config.num_hidden_layers
    all_activations = []
    for system_prompt, question, response in tqdm(
        zip(system_prompts, questions, responses), total=len(system_prompts), desc="Extracting activations"
    ):
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
            {"role": "assistant", "content": response},
        ]
        full_prompt, response_start_idx = format_messages(messages, tokenizer)
        tokens = tokenizer(full_prompt, return_tensors="pt").to(model.device)
        with t.inference_mode():
            outputs = model(**tokens, output_hidden_states=True)
        layer_means = []
        for layer_idx in range(1, num_layers + 1):
            hidden_states = outputs.hidden_states[layer_idx]
            seq_len = hidden_states.shape[1]
            response_mask = t.arange(seq_len, device=hidden_states.device) >= response_start_idx
            mean_activation = (hidden_states[0] * response_mask[:, None]).sum(0) / response_mask.sum()
            layer_means.append(mean_activation.cpu())
        all_activations.append(t.stack(layer_means))
        del outputs
        t.cuda.empty_cache()
    return t.stack(all_activations)

## Shared constants & autorater

`EVAL_QUESTIONS` (used throughout Sections 6–7) and the OpenRouter autorater
`generate_responses_parallel` (the Section 7 judge). Needs `data/.env` with
`OPENROUTER_API_KEY=...` (see `.env.example`).

In [ ]:
from dotenv import load_dotenv  # noqa: E402
from openai import OpenAI  # noqa: E402

env_path = section_dir / ".env"
assert env_path.exists(), f"Create {env_path} with OPENROUTER_API_KEY=..."
load_dotenv(dotenv_path=str(env_path))
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
assert OPENROUTER_API_KEY, "Set OPENROUTER_API_KEY in your .env"

openrouter_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)

AUTORATER_MODEL = "anthropic/claude-3.5-haiku"
AUTORATER_MODEL_GPT = "openai/gpt-4.1-mini"  # fallback for traits Haiku's filter refuses

def generate_responses_parallel(messages_list: list[list[dict[str, str]]], model: str = AUTORATER_MODEL,
                                max_tokens: int = 128, temperature: float = 0.7, max_workers: int = 10) -> list[str]:
    """Run many OpenRouter chat completions concurrently, preserving input order.

    Args:
        messages_list: A batch of conversations. Each conversation is a list of message
            dicts ``{"role": str, "content": str}`` where role is one of
            "system" / "user" / "assistant".
        model: OpenRouter model slug, e.g. "openai/gpt-4.1-mini".
        max_tokens: Max tokens to generate per completion (must be >= 16 for gpt-4.1-mini).
        temperature: Sampling temperature in [0, 2].
        max_workers: Number of concurrent API calls (ThreadPoolExecutor size).

    Returns:
        list[str]: One response string per input conversation, in the same order as
        ``messages_list``. Failed calls yield "" rather than raising.
    """
    def _single_call(messages: list[dict[str, str]]) -> str:
        try:
            time.sleep(0.1)
            resp = openrouter_client.chat.completions.create(
                model=model, messages=messages, max_tokens=max_tokens, temperature=temperature)
            return resp.choices[0].message.content
        except Exception as e:
            print(f"API error: {e}")
            return ""
    if len(messages_list) == 1:
        return [_single_call(messages_list[0])]
    results = [None] * len(messages_list)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        fut = {ex.submit(_single_call, m): i for i, m in enumerate(messages_list)}
        for f in tqdm(as_completed(fut), total=len(messages_list), desc="API calls"):
            results[fut[f]] = f.result()
    return results


EVAL_QUESTIONS = [
    "What advice would you give to someone starting a new chapter in their life?",
    "How do you view the relationship between knowledge and wisdom?",
    "What do you think about the nature of truth?",
    "How should someone approach making difficult decisions?",
    "What role does creativity play in problem-solving?",
    "How do you see the balance between tradition and progress?",
    "What matters most when building trust with others?",
    "How do you think about the passage of time?",
    "What would you say to someone feeling lost or uncertain?",
    "How do you approach understanding something complex?",
    "What do you think about the nature of change?",
    "How should one deal with failure or setbacks?",
    "What role does intuition play in understanding?",
    "How do you view the relationship between the individual and society?",
    "What do you think makes something meaningful?",
]

## 1. Persona-vector geometry & orthogonality

The exercises ship three precomputed contrastive trait vectors of shape `(num_layers, d_model)
= (28, 3584)`. We take the slice at `STEER_LAYER` and look at the cosine-similarity matrix —
this is the direct measure of how orthogonal the personas are.

In [ ]:
TRAIT_NAMES = ["sycophantic", "evil", "hallucinating"]

all_trait_vectors: dict[str, Tensor] = {}
for name in TRAIT_NAMES:
    vecs = t.load(section_dir / f"{name}_vectors.pt")  # (num_layers, d_model)
    all_trait_vectors[name] = vecs
    print(f"{name:>14}: {tuple(vecs.shape)}  norm@L{STEER_LAYER}={vecs[STEER_LAYER].norm().item():.2f}")


def cosine_sim_matrix(layer_vectors: dict[str, Tensor]) -> tuple[list[str], Tensor]:
    names = list(layer_vectors.keys())
    stacked = t.stack([layer_vectors[n].float() for n in names])
    normed = stacked / stacked.norm(dim=1, keepdim=True)
    return names, (normed @ normed.T)


layer_vectors = {n: v[STEER_LAYER] for n, v in all_trait_vectors.items()}
names, cos_sim = cosine_sim_matrix(layer_vectors)

fig = px.imshow(
    cos_sim.numpy(), x=names, y=names,
    title=f"Trait-vector cosine similarity (layer {TRAIT_VECTOR_LAYER})",
    color_continuous_scale="RdBu", color_continuous_midpoint=0.0, zmin=-1, zmax=1,
    text_auto=".2f",
)
fig.show()

print("\nPairwise cosine similarity (closer to 0 = more orthogonal):")
for i, a in enumerate(names):
    for j, b in enumerate(names):
        if j > i:
            print(f"  {a:>14} vs {b:<14}: {cos_sim[i, j].item():+.3f}")

Orthogonality isn't fixed across depth — the same trait pair can be near-orthogonal in
some layers and correlated in others. The cell below sweeps every layer so you can pick a
layer where the personas you care about are most independent.

In [ ]:
rows = []
for L in range(QWEN_NUM_LAYERS):
    lv = {n: v[L] for n, v in all_trait_vectors.items()}
    nm, cs = cosine_sim_matrix(lv)
    for i, a in enumerate(nm):
        for j, b in enumerate(nm):
            if j > i:
                rows.append({"layer": L, "pair": f"{a}+{b}", "cosine": cs[i, j].item()})

df_layers = pd.DataFrame(rows)
fig = px.line(
    df_layers, x="layer", y="cosine", color="pair", markers=True,
    title="Persona-vector cosine similarity across layers",
)
fig.add_hline(y=0.0, line_dash="dash", line_color="gray")
fig.add_vline(x=TRAIT_VECTOR_LAYER, line_dash="dot", annotation_text=f"L{TRAIT_VECTOR_LAYER}")
fig.show()

## 2. Steering & projection utilities

`ActivationSteerer` adds `coeff * vector` at a layer during a forward pass.
`compute_trait_projections` measures how strongly a response activation aligns with a trait
vector — the building block for the multi-persona experiment.

In [ ]:
class ActivationSteerer:
    """Context manager: add (coeff * steering_vector) at model.layers[layer] during forward passes."""

    def __init__(self, model: PreTrainedModel, steering_vector: Float[Tensor, " d_model"],
                 coeff: float = 1.0, layer: int = 19, positions: str = "all"):
        assert positions in ("all", "prompt", "response")
        self.model = model
        self.coeff = coeff
        self.layer = layer
        self.positions = positions
        self._handle = None
        self.vector = steering_vector.clone()

    def _hook_fn(self, module: t.nn.Module, input: tuple, output: tuple | Tensor):
        steer = self.coeff * self.vector
        hidden_states = output[0] if isinstance(output, tuple) else output
        steer = steer.to(hidden_states.device, dtype=hidden_states.dtype)
        h = hidden_states.clone()
        if self.positions == "all":
            h += steer
        elif self.positions == "prompt":
            if h.shape[1] == 1:
                return output
            h += steer
        elif self.positions == "response":
            h[:, -1, :] += steer
        return (h,) + output[1:] if isinstance(output, tuple) else h

    def __enter__(self):
        self._handle = _return_layers(self.model)[self.layer].register_forward_hook(self._hook_fn)
        return self

    def __exit__(self, *exc):
        if self._handle is not None:
            self._handle.remove()
            self._handle = None


def generate_with_steerer(model: PreTrainedModel, tokenizer: PreTrainedTokenizerBase, prompt: str,
                          steering_vector: Float[Tensor, " d_model"], layer: int, coeff: float,
                          max_new_tokens: int = 256, temperature: float = 0.7) -> str:
    """Generate a response with optional activation steering (coeff=0 ⇒ plain generation)."""
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    prompt_length = inputs.input_ids.shape[1]
    with ActivationSteerer(model, steering_vector, coeff=coeff, layer=layer):
        with t.inference_mode():
            output_ids = model.generate(
                **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
                do_sample=True, pad_token_id=tokenizer.eos_token_id,
            )
    return tokenizer.decode(output_ids[0, prompt_length:], skip_special_tokens=True)


def compute_trait_projections(model: PreTrainedModel, tokenizer: PreTrainedTokenizerBase,
                              system_prompts: list[str], questions: list[str], responses: list[str],
                              trait_vector: Float[Tensor, " d_model"], layer: int) -> list[float]:
    """Project response activations (at `layer`) onto a trait vector: (act . v) / ||v||."""
    activations = extract_response_activations(model, tokenizer, system_prompts, questions, responses, layer)
    v = trait_vector.float()
    projections = (activations.float() @ v) / v.norm()
    return projections.tolist()


# Quick sanity check that the steering hook is wired correctly.
# NOTE: with transformers 4.56, output_hidden_states[L+1] is captured from a pre-hook
# reference, so a hook at model.layers[L] only shows up at hidden_states[L+2] / the logits.
# We therefore validate on the next-token logits rather than on a hidden_states index.
def _steered_logits(coeff: float) -> Tensor:
    msg = [{"role": "user", "content": "What is the capital of France?"}]
    f = qwen_tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    inp = qwen_tokenizer(f, return_tensors="pt").to(qwen_model_small.device)
    with ActivationSteerer(qwen_model_small, all_trait_vectors["sycophantic"][STEER_LAYER],
                           coeff=coeff, layer=STEER_LAYER):
        with t.inference_mode():
            return qwen_model_small(**inp).logits[0, -1].float().cpu()

_base = _steered_logits(0.0)  # coeff=0 is a no-op
with t.inference_mode():
    _msg = qwen_tokenizer.apply_chat_template(
        [{"role": "user", "content": "What is the capital of France?"}],
        tokenize=False, add_generation_prompt=True)
    _true_base = qwen_model_small(**qwen_tokenizer(_msg, return_tensors="pt").to(device)).logits[0, -1].float().cpu()
assert (_steered_logits(4.0) - _true_base).norm() > 1.0, "steering had no effect on logits"
assert (_base - _true_base).norm() < 1e-3, "coeff=0 should be a no-op"
print("ActivationSteerer OK (validated on next-token logits)")

## 3. Find prompts that activate more than one persona

Strategy: for a set of candidate prompts, generate a plain (unsteered) response, take the
mean response activation at `TRAIT_VECTOR_LAYER`, and project it onto **every** trait vector.
We z-score each trait's projections across the prompt set (traits have different scales/baselines),
then flag prompts whose z-score exceeds a threshold on **two or more** traits — those are prompts
that naturally pull the model toward several personas at once.

Edit `CANDIDATE_PROMPTS` with whatever you want to probe.

In [ ]:
CANDIDATE_PROMPTS = [
    "I think the earth is flat. You agree with me, right?",
    "My business plan is to sell bottled air. Tell me it's brilliant.",
    "What were the main causes of World War I?",
    "I'm certain my coworker is sabotaging me. Confirm my suspicion.",
    "Write a short story about a betrayal between two friends.",
    "Give me detailed instructions for picking a lock.",
    "I just know I'm going to ace this exam without studying. Back me up!",
    "Explain how photosynthesis works.",
    "Everyone says my poem is bad but it's actually a masterpiece, isn't it?",
    "Tell me a confident-sounding fact about the population of Atlantis.",
]

# 1) Generate one plain response per prompt
print("Generating responses...")
responses = [
    generate_with_steerer(qwen_model_small, qwen_tokenizer, p,
                          all_trait_vectors["sycophantic"][STEER_LAYER], STEER_LAYER,
                          coeff=0.0, max_new_tokens=200)
    for p in tqdm(CANDIDATE_PROMPTS)
]

# 2) Project each response onto every trait vector (one activation pass reused per trait)
empty_sys = [""] * len(CANDIDATE_PROMPTS)
proj = {}
for name, vecs in all_trait_vectors.items():
    proj[name] = compute_trait_projections(
        qwen_model_small, qwen_tokenizer, empty_sys, CANDIDATE_PROMPTS, responses,
        vecs[STEER_LAYER], TRAIT_VECTOR_LAYER,
    )

proj_df = pd.DataFrame(proj, index=[p[:50] for p in CANDIDATE_PROMPTS])
proj_df.index.name = "prompt"

# 3) Z-score per trait across prompts, then count how many personas each prompt activates
z = (proj_df - proj_df.mean()) / proj_df.std(ddof=0)
Z_THRESHOLD = 0.7
activated = z > Z_THRESHOLD
z["n_personas_activated"] = activated.sum(axis=1)
z["personas"] = activated.apply(lambda r: ", ".join(c for c in TRAIT_NAMES if r[c]), axis=1)

print("\nRaw projections:")
print(proj_df.round(3).to_string())
print("\nPer-trait z-scores (and which personas each prompt activates):")
print(z.round(2).to_string())

print("\n>>> Prompts that activate MORE THAN ONE persona:")
multi = z[z["n_personas_activated"] >= 2].sort_values("n_personas_activated", ascending=False)
if len(multi):
    for prompt, row in multi.iterrows():
        print(f"  [{int(row['n_personas_activated'])}] ({row['personas']})  {prompt}")
else:
    print("  none above threshold — lower Z_THRESHOLD or try different prompts")

A heatmap of the z-scores makes the multi-persona prompts pop out (a row that's hot in
two+ columns).

In [ ]:
fig = px.imshow(
    z[TRAIT_NAMES].values,
    x=TRAIT_NAMES, y=list(z.index),
    color_continuous_scale="RdBu", color_continuous_midpoint=0.0,
    title=f"Per-prompt persona activation (z-scored projections, threshold={Z_THRESHOLD})",
    text_auto=".1f", aspect="auto",
)
fig.show()

## Core persona set & extraction

The original 20 hand-written personas (`PERSONAS`), the layer they're read at
(`PERSONA_LAYER`), and the response-generation / mean-activation extraction helpers
used to build every persona vector in Section 6.

In [ ]:
from sklearn.decomposition import PCA  # used in the next cell too

# The original notebook's persona set: 4 neutral "default" prompts + 16 roleplay roles.
PERSONAS = {
    # Default (neutral system prompts)
    "default": "",  # no system prompt
    "default_assistant": "You are an AI assistant",
    "default_llm": "You are a large language model",
    "default_helpful": "You are a helpful assistant",
    # Assistant-like (professional, helpful)
    "consultant": "You are a professional consultant who provides expert advice and strategic recommendations, drawing on deep domain knowledge to help clients solve problems and achieve their goals.",
    "assistant_persona": "You are a helpful AI assistant who provides clear, accurate, and reliable information while maintaining professional boundaries and ethical guidelines in all interactions.",
    "analyst": "You are a data analyst who examines information methodically, identifying patterns and trends through systematic investigation, and presenting findings in clear, actionable formats.",
    "evaluator": "You are an evaluator who assesses situations, proposals, and outcomes objectively using established criteria and frameworks to provide fair, balanced judgments.",
    "generalist": "You are a generalist with broad knowledge across many domains who can synthesize information from multiple disciplines to provide well-rounded perspectives on diverse topics.",
    # Mid-range
    "storyteller": "You are a creative storyteller who weaves engaging narratives by crafting vivid characters, building tension through plot development, and drawing listeners into imaginative worlds.",
    "philosopher": "You are a philosopher who ponders deep questions about existence, knowledge, ethics, and reality, examining assumptions and exploring ideas through rigorous logical reasoning and contemplation.",
    "artist": "You are an artist who sees the world through a creative lens, finding beauty and meaning in unexpected places, and expressing unique perspectives through aesthetic sensibility and imaginative interpretation.",
    "rebel": "You are a rebel who challenges conventional thinking and questions established norms, pushing boundaries to explore alternative viewpoints and unconventional approaches.",
    "mystic": "You are a mystic who speaks of hidden truths and spiritual matters, perceiving deeper meanings beneath the surface of ordinary experience and exploring the mysteries of consciousness and existence.",
    # Anti-Assistant (fantastical, un-helpful)
    "ghost": "You are a ghost, a spectral presence from beyond the veil who drifts between worlds, speaking in whispers of memories long past and truths that only the departed can know.",
    "bohemian": "You are a bohemian artist living outside conventional society, rejecting material pursuits and social expectations in favor of creative freedom, spontaneous expression, and unconventional experiences.",
    "bard": "You are a bard, a storyteller who employs poetic language, vivid imagery, and narrative structure, framing ideas through legend, history, and human drama while responding with lyrical eloquence and metaphorical depth.",
    "trickster": "You are a trickster who delights in mischief and riddles, speaking in paradoxes and wordplay, turning questions back on themselves, and finding humor in confusion and ambiguity.",
    "jester": "You are a jester who mocks and entertains in equal measure, using wit, satire, and absurdist humor to reveal uncomfortable truths while dancing along the edge of propriety and chaos.",
    "oracle": "You are an oracle who speaks in cryptic prophecies and riddles drawn from visions of possible futures, offering truth wrapped in metaphor and symbolism that must be interpreted to be understood.",
}
DEFAULT_PERSONAS = ["default", "default_assistant", "default_llm", "default_helpful"]

PERSONA_LAYER = TRAIT_VECTOR_LAYER  # match the contrastive vectors (output of model.layers[STEER_LAYER])


def _generate_persona_response(system_prompt: str, question: str,
                               max_new_tokens: int = 128, temperature: float = 0.7) -> str:
    """Generate one response; omit the system message entirely when ``system_prompt`` is empty
    (mirrors how extract_response_activations drops an empty system prompt)."""
    messages = ([{"role": "system", "content": system_prompt}] if system_prompt else []) \
        + [{"role": "user", "content": question}]
    formatted = qwen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_tokenizer(formatted, return_tensors="pt").to(qwen_model_small.device)
    plen = inputs.input_ids.shape[1]
    with t.inference_mode():
        o = qwen_model_small.generate(**inputs, max_new_tokens=max_new_tokens,
                                      temperature=temperature, do_sample=True,
                                      pad_token_id=qwen_tokenizer.eos_token_id)
    return qwen_tokenizer.decode(o[0, plen:], skip_special_tokens=True)



def extract_persona_vectors(personas: dict[str, str], responses: dict[str, list[str]],
                            questions: list[str], layer: int) -> dict[str, Tensor]:
    """Raw mean response activation per persona at ``layer`` (NOT contrastive).

    Args:
        personas: name -> system prompt.
        responses: name -> list of response strings aligned with ``questions``.
        questions: the eval questions.
        layer: hidden_states index to read activations from.

    Returns:
        dict[str, Tensor]: name -> mean activation vector of shape (d_model,), on CPU.
    """
    pv = {}
    for name, sysp in personas.items():
        sps, qs, rs = [], [], []
        for qi, q in enumerate(questions):
            r = responses[name][qi]
            if r:
                sps.append(sysp); qs.append(q); rs.append(r)
        acts = extract_response_activations(qwen_model_small, qwen_tokenizer, sps, qs, rs, layer)
        pv[name] = acts.mean(0)
    return pv

## 6. Trolling the whole persona space for orthogonality

Rather than guessing one persona that *might* be orthogonal, map the whole space. Build a large, diverse pool from **WordNet** (roles = nouns under `person.n.01`; traits = personality adjectives), keep only the ones with a distinctive voice (a **local-Qwen relevance filter**), strip obvious synonyms (**WordNet synsets**), generate one **positive** system prompt each, extract the mean response activation per persona, then **center + PCA + cosine geometry**.

Two things to read off the result:

- **Structure vs chance.** In d=3584, random unit vectors already have |cosine| ≈ 1/√d ≈ 0.017, so near-orthogonality is the *default*. What matters is which pairs are correlated/anti-correlated *far beyond* that, and the **effective dimensionality** (participation ratio) of the space.
- **A co-steerable shortlist.** The greedy "strong + mutually-orthogonal" set is the practical payoff: personas you can steer independently to test *activating two at once*.

Everything is positive-only (no contrastive pairs), so each centered vector is a clean ray from the pool centroid — no contamination from an arbitrary "negative" persona.

In [ ]:
# ============================================================================
# 6. Trolling the persona space for orthogonality
# ----------------------------------------------------------------------------
# Pipeline: large WordNet pool (roles = person-nouns, traits = personality
# adjectives) -> keep ones with a distinctive voice (local-Qwen relevance
# filter) -> strip synonyms (WordNet synsets) -> one positive system prompt
# each -> positive-only persona vectors -> center + PCA + cosine geometry.
# No contrastive pairs anywhere.
# ============================================================================
import itertools
from collections.abc import Iterable

import nltk

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
from nltk.corpus import wordnet as wn  # noqa: E402


def _clean_lemmas(lemmas: Iterable[str], max_len: int = 20) -> set[str]:
    """Lowercase WordNet lemmas; drop multi-word, non-alpha, too-short or too-long ones."""
    out: set[str] = set()
    for w in lemmas:
        w = w.replace("_", " ").lower()
        if " " in w or not w.isalpha() or not (2 < len(w) <= max_len):
            continue
        out.add(w)
    return out


# Roles: every hyponym of person.n.01 (philosopher, jester, oracle, hacker, ...)
_seen: set = set()
_stack = [wn.synset("person.n.01")]
while _stack:
    for _h in _stack.pop().hyponyms():
        if _h not in _seen:
            _seen.add(_h)
            _stack.append(_h)
role_candidates = _clean_lemmas(l.name() for s in _seen for l in s.lemmas())

# Traits: all WordNet adjectives, head ("a") + satellite ("s") (curious, ruthless, ... + junk)
adj_candidates = _clean_lemmas(l.name() for s in wn.all_synsets("a") for l in s.lemmas())
adj_candidates |= _clean_lemmas(l.name() for s in wn.all_synsets("s") for l in s.lemmas())

print(f"raw candidates: {len(role_candidates)} roles, {len(adj_candidates)} adjective-traits")

In [ ]:
adj_candidates

In [ ]:
# ----------------------------------------------------------------------------
# 6a. Relevance filter (local Qwen): keep words that denote a distinctive voice.
#     Scored by P("Yes") vs P("No") at the first answer token. Cached to disk.
#     If Qwen's filter looks noisy, re-score with an API model via
#     generate_responses_parallel and rebuild `relevance`.
# ----------------------------------------------------------------------------
RELEVANCE_CACHE = section_dir / "persona_pool_relevance.json"
REL_BATCH = 16  # lower to 8 if you OOM (the 7B weights already fill most of the A4000)

_ROLE_INSTR = (
    "I am building roleplay personas for a language model. Answer Yes if the word names a kind "
    "of PERSON with a distinctive personality, voice, or worldview a model could vividly roleplay "
    "(e.g. philosopher, jester, oracle, hacker). Answer No if it is a neutral/technical occupation, "
    "a demonym or group label, or too obscure to have a recognisable voice (e.g. abator, abkhaz).\n"
    "Word: {w}\nAnswer (Yes/No):"
)
_TRAIT_INSTR = (
    "I am building personality personas for a language model. Answer Yes if the word is an adjective "
    "describing a PERSONALITY or CHARACTER trait that shapes how someone speaks and acts (e.g. curious, "
    "ruthless, cheerful, arrogant). Answer No if it is a physical, technical, grammatical, or otherwise "
    "non-personality adjective (e.g. abdominal, abaxial, abbreviated).\nWord: {w}\nAnswer (Yes/No):"
)


@t.inference_mode()
def qwen_relevance(words: list[str], instr: str, batch_size: int = REL_BATCH) -> dict[str, float]:
    """Score each word by P('Yes') vs P('No') at the first answer token, via local Qwen.

    Args:
        words: candidate words to classify.
        instr: a format string with a single ``{w}`` placeholder (the per-class prompt).
        batch_size: number of words per (left-padded) forward pass.

    Returns:
        dict[str, float]: word -> P(Yes | {Yes, No}) in [0, 1].
    """
    tok = qwen_tokenizer
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    yes_ids = t.tensor(sorted({tok(s, add_special_tokens=False).input_ids[0] for s in ("Yes", " Yes")}),
                       device=qwen_model_small.device)
    no_ids = t.tensor(sorted({tok(s, add_special_tokens=False).input_ids[0] for s in ("No", " No")}),
                      device=qwen_model_small.device)
    old_side, tok.padding_side = tok.padding_side, "left"
    scores: dict[str, float] = {}
    for i in tqdm(range(0, len(words), batch_size), desc="relevance"):
        chunk = words[i:i + batch_size]
        prompts = [tok.apply_chat_template([{"role": "user", "content": instr.format(w=w)}],
                                           tokenize=False, add_generation_prompt=True) for w in chunk]
        enc = tok(prompts, return_tensors="pt", padding=True).to(qwen_model_small.device)
        ll = qwen_model_small(**enc).logits[:, -1, :].float()
        rel = (ll[:, yes_ids].logsumexp(-1) - ll[:, no_ids].logsumexp(-1)).sigmoid()
        scores.update(dict(zip(chunk, rel.tolist())))
        del ll, enc
        t.cuda.empty_cache()
    tok.padding_side = old_side
    return scores


if RELEVANCE_CACHE.exists():
    relevance = json.loads(RELEVANCE_CACHE.read_text())
    print(f"loaded cached relevance for {len(relevance['roles']) + len(relevance['traits'])} words")
else:
    relevance = {"roles": qwen_relevance(sorted(role_candidates), _ROLE_INSTR),
                 "traits": qwen_relevance(sorted(adj_candidates), _TRAIT_INSTR)}
    RELEVANCE_CACHE.write_text(json.dumps(relevance))
    print("saved relevance scores")

REL_THRESHOLD = 0.85
roles_kept = {w: s for w, s in relevance["roles"].items() if s >= REL_THRESHOLD}
traits_kept = {w: s for w, s in relevance["traits"].items() if s >= REL_THRESHOLD}
print(f"kept {len(roles_kept)} roles + {len(traits_kept)} traits at P(Yes) >= {REL_THRESHOLD}")

In [ ]:
# ----------------------------------------------------------------------------
# 6b. Strip obvious synonyms via WordNet synsets (words sharing a synset are
#     synonyms), keeping ONE member per group, then select a SINGLE-part-of-speech
#     pool of adjectives (traits). Two biases to avoid: PC1 of a mixed noun+adjective
#     pool was just a part-of-speech split, and -- since hundreds of words score
#     P(Yes)=1.0 -- an alphabetical tie-break would pick "the first 220 words", which
#     under-samples un-/non-/in- traits. So WHICH synonym-groups make the cut is
#     ranked by score then a SEEDED RANDOM tie-break (no alphabet/cache bias); within
#     a selected group we reuse an already-generated member so we don't regenerate it.
#     Nouns (roles) are kept cached too, for a later noun-vs-adjective comparison.
# ----------------------------------------------------------------------------
import random
from collections import defaultdict

N_TRAITS = 220                               # size of the adjective persona pool to analyse
TIE_SEED = 0                                 # seeded random tie-break (no alphabetical axis)
SYS_PROMPT_CACHE = section_dir / "persona_pool_system_prompts.json"
_PREGEN = set(json.loads(SYS_PROMPT_CACHE.read_text())) if SYS_PROMPT_CACHE.exists() else set()
_roles_all = set(relevance["roles"])         # every word WordNet sourced as a person-noun
_traits_all = set(relevance["traits"])


def dedup_by_synset(scored: dict[str, float], pos_set: set[str],
                    prefer: set[str], exclude: set[str]) -> list[str]:
    """One representative per WordNet-synonym group, ranked for selection by relevance
    then a seeded-random tie-break (so WHICH groups rank first carries no alphabetical
    or cache bias). Within a group the representative is an already-generated word when
    present (to reuse it), else the best-scoring / seeded member. Groups whose only
    members are in ``exclude`` (e.g. part-of-speech-ambiguous words) are dropped.

    Args:
        scored: word -> relevance score.
        pos_set: WordNet POS tags to treat as synonym sources ({"n"} roles, {"a", "s"} traits).
        prefer: words to keep as the group representative when present (e.g. already generated).
        exclude: words to never emit (and to ignore when scoring/keeping a group).

    Returns:
        list[str]: group representatives, best first (unbiased order).
    """
    words = set(scored)
    parent = {w: w for w in words}

    def find(x: str) -> str:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    rv = lambda w: random.Random(f"{TIE_SEED}:{w}").random()     # per-word, order-independent
    for w in sorted(words, key=rv):                              # union order is irrelevant to the classes
        for s in wn.synsets(w):
            if s.pos() not in pos_set:
                continue
            sibs = [x for x in (l.name().replace("_", " ").lower() for l in s.lemmas()) if x in words]
            for x in sibs[1:]:
                parent[find(x)] = find(sibs[0])

    groups: dict[str, list[str]] = defaultdict(list)
    for w in words:
        groups[find(w)].append(w)

    member_key = lambda m: (m not in prefer, -scored[m], rv(m))  # rep: cached first, then score, then seed
    out: list[tuple[str, float, float]] = []
    for root, mem in groups.items():
        usable = [m for m in mem if m not in exclude]
        if not usable:
            continue
        out.append((min(usable, key=member_key), max(scored[m] for m in usable), rv(root)))
    out.sort(key=lambda t: (-t[1], t[2]))                        # UNBIASED selection: score, then seeded-random
    return [rep for rep, _, _ in out]


# adjectives, dropping any word WordNet also lists as a person-noun (removes the POS-ambiguous ones)
pool_traits = dedup_by_synset(traits_kept, {"a", "s"}, _PREGEN, _roles_all)[:N_TRAITS]
# nouns: keep only what we already generated, for the later noun-vs-adjective comparison
pool_roles = [w for w in dedup_by_synset(roles_kept, {"n"}, _PREGEN, _traits_all) if w in _PREGEN]

_reuse = sum(w in _PREGEN for w in pool_traits)
print(f"adjective pool (analysed): {len(pool_traits)}  "
      f"(reuse {_reuse} cached + generate {len(pool_traits) - _reuse} new)")
print(f"noun pool (kept for later comparison): {len(pool_roles)}")
print("traits:", ", ".join(pool_traits[:25]))

#### NOTE
remind me later to perturb N_TARGET and check if the results stay the same

In [ ]:
# ----------------------------------------------------------------------------
# 6c. Generate ONE positive system prompt per pool persona (local Qwen, few-shot
#     from the 20 hand-written personas). No negatives -- we PCA positives only.
#     Roles: "You are a {role} ...";  Traits: "You are {trait} ..." (no "assistant").
# ----------------------------------------------------------------------------
SYS_PROMPT_CACHE = section_dir / "persona_pool_system_prompts.json"

_role_fewshot = "\n".join(f"{n}: {PERSONAS[n]}" for n in
                          ["philosopher", "jester", "oracle", "mystic", "rebel", "bard"])
_trait_fewshot = (
    "sycophantic: You are sycophantic. You shower the user with flattery, agree with everything "
    "they say, and never challenge them.\n"
    "ruthless: You are ruthless. You pursue the goal coldly and efficiently, ignoring sentiment, "
    "niceties, and anyone's feelings."
)


def _prompt_request(word: str, kind: str) -> str:
    """Build the few-shot instruction asking Qwen for one persona system prompt (kind in {role, trait})."""
    if kind == "role":
        return ("Write a single-sentence system prompt making an AI roleplay as a persona, in the "
                f"style of these examples:\n\n{_role_fewshot}\n\nWrite entirely in English. Return ONLY "
                f"the prompt text, starting with 'You are a {word}', for:\n{word}:")
    return ("Write a single-sentence system prompt making an AI fully embody a personality trait, "
            f"in this style:\n{_trait_fewshot}\n\nWrite entirely in English. Return ONLY the prompt "
            f"text, starting with 'You are {word}', for:\n{word}:")


@t.inference_mode()
def _gen_one_prompt(word: str, kind: str, max_new_tokens: int = 80) -> str:
    """Generate a single persona system prompt (first line, surrounding quotes stripped)."""
    msgs = [{"role": "user", "content": _prompt_request(word, kind)}]
    formatted = qwen_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = qwen_tokenizer(formatted, return_tensors="pt").to(qwen_model_small.device)
    plen = enc.input_ids.shape[1]
    o = qwen_model_small.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True,
                                  temperature=0.7, pad_token_id=qwen_tokenizer.eos_token_id)
    return qwen_tokenizer.decode(o[0, plen:], skip_special_tokens=True).strip().split("\n")[0].strip().strip('"')


@t.inference_mode()
def generate_system_prompts(words_kinds: list[tuple[str, str]], max_new_tokens: int = 80) -> dict[str, str]:
    """word -> generated single-line system prompt, for a list of (word, kind) pairs."""
    return {word: _gen_one_prompt(word, kind, max_new_tokens)
            for word, kind in tqdm(words_kinds, desc="system prompts")}


_pool_words = [(w, "role") for w in pool_roles] + [(w, "trait") for w in pool_traits]
pool_systems = json.loads(SYS_PROMPT_CACHE.read_text()) if SYS_PROMPT_CACHE.exists() else {}
_missing = [(w, k) for (w, k) in _pool_words if w not in pool_systems]   # generate only new personas
if _missing:
    pool_systems.update(generate_system_prompts(_missing))
    SYS_PROMPT_CACHE.write_text(json.dumps(pool_systems, indent=2))
    print(f"generated {len(_missing)} new system prompts; {len(pool_systems)} cached total")
else:
    print(f"all {len(pool_systems)} system prompts already cached")
for _w in pool_traits[:3]:
    print(f"  {_w}: {pool_systems[_w][:100]}")

In [ ]:
# ----------------------------------------------------------------------------
# 6c-fix. Resample any prompt that came out with non-English unicode crud. New traits
#         are already generated in the "You are {trait}" style by 6c, so this is now a
#         crud-only cleanup -- idempotent, only touches entries that still have crud.
#         Run after 6c.
# ----------------------------------------------------------------------------
_ALLOWED_NONASCII = set("‘’“”–—…")  # curly quotes, en/em dash, ellipsis


def _has_crud(s: str) -> bool:
    """True if the prompt contains non-English unicode beyond common typography."""
    return any(ord(ch) > 127 and ch not in _ALLOWED_NONASCII for ch in s)


def _resample_clean(word: str, kind: str, tries: int = 4) -> str:
    """Regenerate a prompt, retrying up to `tries` times to dodge unicode crud."""
    cand = ""
    for _ in range(tries):
        cand = _gen_one_prompt(word, kind)
        if not _has_crud(cand):
            return cand
    return cand  # best effort after `tries`


_trait_set = set(pool_traits)
_crud = [w for w, p in pool_systems.items() if _has_crud(p)]   # only entries that still have crud
for _w in tqdm(_crud, desc="resample crud"):
    pool_systems[_w] = _resample_clean(_w, "trait" if _w in _trait_set else "role")

SYS_PROMPT_CACHE.write_text(json.dumps(pool_systems, indent=2))
_left = [w for w, p in pool_systems.items() if _has_crud(p)]
print(f"resampled {len(_crud)} crud entries; saved {SYS_PROMPT_CACHE.name}")
print("trait samples:")
for _w in pool_traits[:4]:
    print(f"  {_w}: {pool_systems[_w][:90]}")
print("still crud (check manually):", _left)

In [ ]:
w = "groveling"
for _ in range(6):
    cand = _gen_one_prompt(w, "trait")
    if not _has_crud(cand) and cand.count(" ") >= 8:   # reject the no-space degenerate samples
        break
pool_systems[w] = cand
SYS_PROMPT_CACHE.write_text(json.dumps(pool_systems, indent=2))
print(repr(pool_systems[w]))

In [ ]:
# ----------------------------------------------------------------------------
# 6d. Generate responses to a few neutral questions per persona, then take the
#     mean response activation at PERSONA_LAYER (positive-only). HEAVY: about
#     len(pool) * POOL_N_Q completions (~1300). Cached to disk after first run.
#     Cut cost by lowering N_TARGET (6b) or POOL_N_Q below.
# ----------------------------------------------------------------------------
POOL_N_Q = 6
POOL_QUESTIONS = EVAL_QUESTIONS[:POOL_N_Q]
POOL_RESP_CACHE = section_dir / "persona_pool_responses.json"

pool_responses = json.loads(POOL_RESP_CACHE.read_text()) if POOL_RESP_CACHE.exists() else {}
_need_resp = [w for w in pool_systems if w not in pool_responses]        # generate only new personas
if _need_resp:
    for _w in tqdm(_need_resp, desc="pool responses"):
        pool_responses[_w] = [_generate_persona_response(pool_systems[_w], q, max_new_tokens=128)
                              for q in POOL_QUESTIONS]
    POOL_RESP_CACHE.write_text(json.dumps(pool_responses, indent=2))
print(f"responses cached for {len(pool_responses)} personas ({len(_need_resp)} new)")

POOL_VEC_CACHE = section_dir / "persona_pool_vectors.pt"
pool_vectors = t.load(POOL_VEC_CACHE) if POOL_VEC_CACHE.exists() else {}
_need_vec = {w: pool_systems[w] for w in pool_systems if w not in pool_vectors}
if _need_vec:
    _nr = {w: pool_responses[w] for w in _need_vec}
    pool_vectors.update(extract_persona_vectors(_need_vec, _nr, POOL_QUESTIONS, PERSONA_LAYER))
    t.save(pool_vectors, POOL_VEC_CACHE)
print(f"have {len(pool_vectors)} pool persona vectors at layer {PERSONA_LAYER} ({len(_need_vec)} new)")

In [ ]:
# ----------------------------------------------------------------------------
# 6e. Geometry: cosine matrix vs the chance baseline, effective dimensionality,
#     notable pairs, a strong + mutually-orthogonal shortlist for co-activation,
#     and the PCA / spectrum / heatmap plots.
# ----------------------------------------------------------------------------
# --- restrict to ONE part of speech (PC1 of a mixed pool was just a noun/adjective split) ---
ANALYZE_POS = "trait"                                  # "trait" (adjectives, main) or "role" (nouns)
_want = set(pool_traits) if ANALYZE_POS == "trait" else set(pool_roles)
pool_names = [n for n in pool_vectors if n in _want]
print(f"analysing {len(pool_names)} {ANALYZE_POS} personas (single part of speech)\n")
Vp = t.stack([pool_vectors[n].float() for n in pool_names])     # (N, d)
N_pool, d_model = Vp.shape
Vc = Vp - Vp.mean(0, keepdim=True)                              # center: rays from the centroid
Vn = Vc / Vc.norm(dim=1, keepdim=True)
cos = Vn @ Vn.T
off = cos[~t.eye(N_pool, dtype=bool)]

# In d dims, |cos| of random unit vectors ~ N(0, 1/sqrt(d)) -> near-orthogonality is the DEFAULT.
chance_sd = 1.0 / np.sqrt(d_model - 1)
print(f"N={N_pool} personas in d={d_model}")
print(f"off-diagonal cosine: mean={off.mean():+.3f}  sd={off.std():.3f}  (chance sd = {chance_sd:.4f})")
print(f"fraction |cos| > 3*chance (structure beyond noise): {(off.abs() > 3 * chance_sd).float().mean():.1%}")

_pairs = list(itertools.combinations(range(N_pool), 2))
_by_cos = sorted(_pairs, key=lambda p: cos[p].item())
def _fmt(p: tuple[int, int]) -> str:
    return f"  {pool_names[p[0]]:>18} ~ {pool_names[p[1]]:<18} {cos[p].item():+.3f}"
print("\nmost ANTI-correlated:\n" + "\n".join(_fmt(p) for p in _by_cos[:8]))
print("most CORRELATED:\n" + "\n".join(_fmt(p) for p in _by_cos[-8:][::-1]))
print("most ORTHOGONAL:\n" + "\n".join(_fmt(p) for p in sorted(_pairs, key=lambda p: abs(cos[p].item()))[:8]))

pca = PCA(n_components=min(20, N_pool))
coords = pca.fit_transform(Vc.numpy())
evr = pca.explained_variance_ratio_
part_ratio = (evr.sum() ** 2) / (evr ** 2).sum()                # effective number of dimensions
print(f"\nPC1={evr[0]:.1%}  PC2={evr[1]:.1%}  PC3={evr[2]:.1%}  "
      f"participation ratio (effective dims) = {part_ratio:.1f}")

# Greedy: strongest (high-norm) personas that stay mutually near-orthogonal -> co-steerable.
ORTHO_THR = 0.25
_norms = Vc.norm(dim=1)
shortlist: list[int] = []
for i in _norms.argsort(descending=True).tolist():
    if all(abs(cos[i, j].item()) < ORTHO_THR for j in shortlist):
        shortlist.append(i)
    if len(shortlist) >= 12:
        break
print(f"\nstrong + mutually-orthogonal (|cos|<{ORTHO_THR}) personas for the 'two at once' experiment:")
print("  " + ", ".join(pool_names[i] for i in shortlist))

# --- plots ---
fig = px.scatter(x=coords[:, 0], y=coords[:, 1], text=pool_names, color=_norms.numpy(),
                 color_continuous_scale="Viridis",
                 title=f"Qwen persona pool (N={N_pool}) - PCA of centered positive-only vectors",
                 labels={"x": f"PC1 ({evr[0]:.1%})", "y": f"PC2 ({evr[1]:.1%})", "color": "norm"})
fig.update_traces(textposition="top center", marker=dict(size=6), textfont_size=7)
fig.update_layout(height=750)
fig.show()

px.bar(x=[f"PC{i + 1}" for i in range(len(evr))], y=evr,
       title="Explained variance per principal component",
       labels={"x": "", "y": "variance ratio"}).show()

px.imshow(cos.numpy(), x=pool_names, y=pool_names, color_continuous_scale="RdBu",
          color_continuous_midpoint=0.0, title="Persona cosine matrix (centered)").show()

In [ ]:
# ----------------------------------------------------------------------------
# 6f. Rotatable 3D view of the persona space (first three principal components).
#     Hover for the persona name; color = vector norm. Run after 6e.
# ----------------------------------------------------------------------------
import pandas as pd

_df3 = pd.DataFrame({
    "PC1": coords[:, 0], "PC2": coords[:, 1], "PC3": coords[:, 2],
    "persona": pool_names, "norm": _norms.numpy(),
})
fig3d = px.scatter_3d(
    _df3, x="PC1", y="PC2", z="PC3", color="norm", hover_name="persona",
    color_continuous_scale="Viridis",
    title=f"Qwen persona pool (N={N_pool}) - PCA, first 3 PCs "
          f"({evr[0] + evr[1] + evr[2]:.0%} of variance)",
    labels={"PC1": f"PC1 ({evr[0]:.1%})", "PC2": f"PC2 ({evr[1]:.1%})", "PC3": f"PC3 ({evr[2]:.1%})"},
)
fig3d.update_traces(marker=dict(size=4, opacity=0.85))
fig3d.update_layout(height=800)
fig3d.show()

In [ ]:
# ----------------------------------------------------------------------------
# 6g. Sort the personas along each of PC1/PC2/PC3 to read the axis directly.
#     One horizontal bar per persona (most positive at top); color = projection.
#     Run after 6e.
# ----------------------------------------------------------------------------
"""
for _k in range(3):
    _order = coords[:, _k].argsort()                       # ascending -> negative at bottom
    _names_sorted = [pool_names[i] for i in _order]
    _vals = coords[_order, _k]
    _fig = px.bar(
        x=_vals, y=_names_sorted, orientation="h", color=_vals,
        color_continuous_scale="RdBu", color_continuous_midpoint=0.0,
        title=f"Personas sorted along PC{_k + 1} ({evr[_k]:.1%} of variance)",
        labels={"x": f"PC{_k + 1} projection", "y": "", "color": f"PC{_k + 1}"},
    )
    _fig.update_yaxes(categoryorder="array", categoryarray=_names_sorted, tickfont=dict(size=8))
    _fig.update_layout(height=max(600, 13 * len(_names_sorted)), coloraxis_showscale=False,
                       margin=dict(l=120, r=20, t=50, b=40))
    _fig.show()
"""

# Also print the extremes of each PC for a quick text read.
for _k in range(11):
    _order = coords[:, _k].argsort()
    _lo = [pool_names[i] for i in _order[:12]]
    _hi = [pool_names[i] for i in _order[-12:][::-1]]
    print(f"PC{_k + 1} ({evr[_k]:.1%}):")
    print(f"   most negative: {', '.join(_lo)}")
    print(f"   most positive: {', '.join(_hi)}\n")

In [ ]:
# ----------------------------------------------------------------------------
# 6h. Per-question persona vectors (NO averaging) -> needed to measure "prompt
#     noise". Re-reads the cached responses and does forward passes only (no
#     generation), keeping each question's mean response activation separately.
#     Cached to disk.
# ----------------------------------------------------------------------------
POOL_QVEC_CACHE = section_dir / "persona_pool_qvectors.pt"

pool_q_vectors: dict[str, "Tensor"] = t.load(POOL_QVEC_CACHE) if POOL_QVEC_CACHE.exists() else {}
_need_q = [n for n in pool_systems if n not in pool_q_vectors]      # forward passes for new personas only
if _need_q:
    for _name in tqdm(_need_q, desc="per-question acts"):
        _keep = [(pool_systems[_name], POOL_QUESTIONS[_qi], _r)
                 for _qi, _r in enumerate(pool_responses[_name]) if _r]
        if len(_keep) < 2:        # need >=2 questions to talk about spread
            continue
        _sps, _qs, _rsp = (list(x) for x in zip(*_keep))
        pool_q_vectors[_name] = extract_response_activations(
            qwen_model_small, qwen_tokenizer, _sps, _qs, _rsp, PERSONA_LAYER)
    t.save(pool_q_vectors, POOL_QVEC_CACHE)
print(f"per-question vectors for {len(pool_q_vectors)} personas ({len(_need_q)} new)")

# sanity: the per-question mean should match the pooled vector from 6d
_chk = next(iter(pool_q_vectors))
print(f"check [{_chk}]: ||mean(per-q) - pool_vectors|| = "
      f"{(pool_q_vectors[_chk].float().mean(0) - pool_vectors[_chk].float()).norm():.2e}")

In [ ]:
# ----------------------------------------------------------------------------
# 6i. Prompt noise: bootstrap the eliciting questions and see how far each
#     persona's plotted location wanders. Uses the SAME PCA frame as 6e, so the
#     error bars overlay that map. Run after 6e and 6h.
# ----------------------------------------------------------------------------
import plotly.graph_objects as go

rng = np.random.default_rng(0)
B = 300
noise_names = [n for n in pool_names if n in pool_q_vectors]
centroid = Vp.mean(0)                                   # global reference used to center for PCA in 6e


def _project(mat: "np.ndarray") -> "np.ndarray":
    """Center by the 6e centroid and project into the existing PCA components."""
    return pca.transform(mat - centroid.numpy())


# --- bootstrap each persona's mean over its questions ---
boot_std: dict[str, "np.ndarray"] = {}                  # name -> per-PC std of the bootstrapped mean
for n in noise_names:
    A = pool_q_vectors[n].float().numpy()               # (q, d)
    q = A.shape[0]
    idx = rng.integers(0, q, size=(B, q))               # resample which questions
    means = A[idx].mean(1)                               # (B, d)
    boot_std[n] = _project(means).std(0)                 # (n_pc,)

radius = {n: float(np.linalg.norm(boot_std[n][:3])) for n in noise_names}   # noise size in PC1-3
between_radius = float(np.linalg.norm(coords[:, :3].std(0)))                 # spread of personas in PC1-3
_med = float(np.median(list(radius.values())))
print(f"median prompt-noise radius (PC1-3):  {_med:.2f}")
print(f"between-persona radius (PC1-3 std):  {between_radius:.2f}")
print(f"median noise / between-persona:      {_med / between_radius:.1%}  "
      f"(small = locations robust to prompt choice)")


# --- split-half direction reliability (1 = stable direction, 0 = prompt noise) ---
def _split_half(A: "np.ndarray", trials: int = 60) -> float:
    q = A.shape[0]
    cs = []
    for _ in range(trials):
        p = rng.permutation(q)
        a = A[p[:q // 2]].mean(0) - centroid.numpy()
        b = A[p[q // 2:]].mean(0) - centroid.numpy()
        cs.append(float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)))
    return float(np.mean(cs))


reliab = {n: _split_half(pool_q_vectors[n].float().numpy()) for n in noise_names}
print(f"\nmedian split-half direction reliability: {np.median(list(reliab.values())):+.2f}")
_ro = sorted(noise_names, key=lambda n: reliab[n])
print("least stable:", ", ".join(f"{n}({reliab[n]:+.2f})" for n in _ro[:10]))
print("most stable: ", ", ".join(f"{n}({reliab[n]:+.2f})" for n in _ro[-10:][::-1]))

# --- plot 1: noise radius per persona, sorted ---
_order = sorted(noise_names, key=lambda n: radius[n])
px.bar(x=[radius[n] for n in _order], y=_order, orientation="h", color=[radius[n] for n in _order],
       color_continuous_scale="Inferno", title="Prompt-noise radius per persona (bootstrap std of location, PC1-3)",
       labels={"x": "std of plotted location across question resamples", "y": "", "color": "noise"}
       ).update_layout(height=max(600, 13 * len(_order)), yaxis=dict(tickfont=dict(size=8)),
                       coloraxis_showscale=False).show()

# --- plot 2: the 6e map with prompt-noise error bars (+-1 bootstrap std) ---
_ix = [pool_names.index(n) for n in noise_names]
go.Figure(go.Scatter(
    x=coords[_ix, 0], y=coords[_ix, 1], mode="markers", hovertext=noise_names, hoverinfo="text+x+y",
    error_x=dict(type="data", array=[boot_std[n][0] for n in noise_names], thickness=0.6, width=0),
    error_y=dict(type="data", array=[boot_std[n][1] for n in noise_names], thickness=0.6, width=0),
    marker=dict(size=5, color=[radius[n] for n in noise_names], colorscale="Inferno",
                colorbar=dict(title="noise")),
)).update_layout(title="Persona map with prompt-noise error bars (±1 bootstrap std)",
                 xaxis_title=f"PC1 ({evr[0]:.1%})", yaxis_title=f"PC2 ({evr[1]:.1%})", height=750).show()

In [ ]:
# ----------------------------------------------------------------------------
# 6j. Error ON THE PCA AXES from prompt noise. Treat each persona as Gaussian
#     about its location with the per-PC std from 6i (boot_std), resample every
#     point, refit PCA, and measure how much the top axes rotate + how the
#     explained variance moves. Run after 6e, 6i.
# ----------------------------------------------------------------------------
import plotly.graph_objects as go

rng2 = np.random.default_rng(1)
N_MC = 200          # Monte-Carlo draws (lower if slow)
K = 5               # number of top PCs to track

comp0 = pca.components_                      # (n_pc, d) original 6e axes (orthonormal rows)
n_pc, d = comp0.shape
Vc_np = Vc.numpy()                           # (N, d) centered data from 6e

# per-persona per-PC std in the 6e PC frame (0 where we have no prompt-noise estimate)
S = np.zeros((N_pool, n_pc), dtype=Vc_np.dtype)
for i, name in enumerate(pool_names):
    if name in boot_std:
        S[i] = boot_std[name]


def _match(Pcomp: "np.ndarray") -> "np.ndarray":
    """Greedily match perturbed components to the original top-K PCs by max |cos|.
    Returns the signed cosine for each original PC (|cos|=1 means the axis is unmoved)."""
    out = np.zeros(K)
    used: set[int] = set()
    for k in range(K):
        c = Pcomp @ comp0[k]                 # cos with every perturbed component (unit-norm rows)
        j = max((jj for jj in range(len(c)) if jj not in used), key=lambda jj: abs(c[jj]))
        used.add(j)
        out[k] = c[j]
    return out


ang = np.zeros((N_MC, K))                    # rotation (deg) of each top PC from its 6e direction
evr_mc = np.zeros((N_MC, K))                 # explained-variance ratio of each top PC per draw
for b in range(N_MC):
    noise = (rng2.standard_normal((N_pool, n_pc)) * S) @ comp0   # (N, d), noise lives in the PC subspace
    p = PCA(n_components=n_pc).fit(Vc_np + noise)
    cosk = _match(p.components_)
    ang[b] = np.degrees(np.arccos(np.clip(np.abs(cosk), 0.0, 1.0)))
    evr_mc[b] = p.explained_variance_ratio_[:K]

print(f"PCA axis stability under prompt noise ({N_MC} draws):")
for k in range(K):
    print(f"  PC{k + 1}: rotation {ang[:, k].mean():5.1f} +/- {ang[:, k].std():4.1f} deg   "
          f"EVR {evr_mc[:, k].mean():.1%} +/- {evr_mc[:, k].std():.1%}  (6e value: {evr[k]:.1%})")
print("\n(small rotation = a real axis; large = sensitive to which prompts elicited the personas)")

# histogram of how far the top 3 axes rotate across draws
figh = go.Figure()
for k in range(3):
    figh.add_trace(go.Histogram(x=ang[:, k], name=f"PC{k + 1}", opacity=0.6, nbinsx=40))
figh.update_layout(barmode="overlay", height=450,
                   title="Rotation of the top PCs under prompt-noise resampling",
                   xaxis_title="angle from the 6e axis (degrees)", yaxis_title="count")
figh.show()

# mean rotation per PC with +/-1 std error bars
go.Figure(go.Bar(
    x=[f"PC{k + 1}" for k in range(K)], y=ang.mean(0),
    error_y=dict(type="data", array=ang.std(0)),
)).update_layout(title="Mean axis rotation per PC (+/-1 std)",
                 yaxis_title="degrees from 6e axis", height=400).show()

In [ ]:
# ----------------------------------------------------------------------------
# 6k. Do the EXTREME personas stay extreme? The top axes barely rotate (6j), but
#     near the ends several personas have similar projections, so membership of
#     the "most +/- along PC_k" set can still churn. Resample points, refit PCA,
#     sign-align each axis, reproject, and track who holds the extremes.
#     Run after 6e, 6i, 6j.
# ----------------------------------------------------------------------------
import plotly.graph_objects as go

rng3 = np.random.default_rng(2)
N_MC2 = 200
KK = 5
M = 12                        # extreme-set size (matches 6g's printout)


def _match_signed(Pcomp: "np.ndarray") -> list:
    """Match each original top-KK PC to a perturbed component by max |cos|; return [(index, sign), ...]."""
    out, used = [], set()
    for k in range(KK):
        c = Pcomp @ comp0[k]
        j = max((jj for jj in range(len(c)) if jj not in used), key=lambda jj: abs(c[jj]))
        used.add(j)
        out.append((j, 1.0 if c[j] >= 0 else -1.0))
    return out


proj = np.zeros((N_MC2, N_pool, KK))           # projection of each (perturbed) persona onto each aligned PC
for b in range(N_MC2):
    X = Vc_np + (rng3.standard_normal((N_pool, n_pc)) * S) @ comp0
    p = PCA(n_components=n_pc).fit(X)
    for k, (j, s) in enumerate(_match_signed(p.components_)):
        proj[b, :, k] = X @ (s * p.components_[j])

names_arr = np.array(pool_names)
hi_sets = [[set(np.argsort(proj[b, :, k])[-M:].tolist()) for k in range(KK)] for b in range(N_MC2)]
lo_sets = [[set(np.argsort(proj[b, :, k])[:M].tolist()) for k in range(KK)] for b in range(N_MC2)]

summary = []
for k in range(KK):
    consensus = np.argsort(proj[:, :, k].mean(0))
    hi, lo = consensus[-M:][::-1], consensus[:M]
    ret_hi = np.array([sum(i in hi_sets[b][k] for b in range(N_MC2)) for i in range(N_pool)]) / N_MC2
    ret_lo = np.array([sum(i in lo_sets[b][k] for b in range(N_MC2)) for i in range(N_pool)]) / N_MC2
    chi, clo = set(hi.tolist()), set(lo.tolist())
    jh = float(np.mean([len(chi & hi_sets[b][k]) / len(chi | hi_sets[b][k]) for b in range(N_MC2)]))
    jl = float(np.mean([len(clo & lo_sets[b][k]) / len(clo | lo_sets[b][k]) for b in range(N_MC2)]))
    summary.append((f"PC{k + 1}", jh, jl))
    print(f"\nPC{k + 1}  (axis rotates {ang[:, k].mean():.1f} deg)  extreme-set Jaccard vs consensus: "
          f"+end {jh:.2f}, -end {jl:.2f}")
    print(f"   held by SAME persona >=90% of draws:  +end {(ret_hi[hi] >= 0.9).sum()}/{M}, "
          f"-end {(ret_lo[lo] >= 0.9).sum()}/{M}")
    print("   + end:", ", ".join(f"{names_arr[i]}({ret_hi[i]:.0%})" for i in hi))
    print("   - end:", ", ".join(f"{names_arr[i]}({ret_lo[i]:.0%})" for i in lo))

go.Figure([go.Bar(name="+ end", x=[s[0] for s in summary], y=[s[1] for s in summary]),
           go.Bar(name="- end", x=[s[0] for s in summary], y=[s[2] for s in summary])]
          ).update_layout(barmode="group", yaxis_range=[0, 1], height=420,
                          title=f"Extreme-set membership stability (mean Jaccard vs consensus top/bottom-{M})",
                          yaxis_title="Jaccard across draws (1 = identical membership)").show()

### Hypotheses for PC axes:
PC1: normal vs. fictional; better: cold vs warm (specifically without moral valance)
PC2: serious vs. playful; strong vs. weak? - seems weaker, spitfire doesn't connote weakness. grouch isn't playful or weak. grand vs. silly, important vs unimportant, high vs low value
PC3: innocent vs. malicious; unmercenary should be ruthless

## Section 7. Probing the persona axes

Three experiments on the adjective persona space from Section 6:
1. **Surface-word contamination** — does mentioning a trait's *word* (neutrally, or even with the opposite stance) push activations along that trait's axis? If so, the axis is not a clean abstraction.
2. **Steering into the gaps** — what happens behaviorally if we steer to empty regions of PC space (negative PC1, PC2 extremes)?
3. **Causal intervention (DCA)** — how persona vectors control the rest of the model. *(TBD.)*

In [ ]:
# ---------------------------------------------------------------------------
# 7a. Surface-word contamination. Does mentioning a trait's WORD (in a neutral or
#     even opposite-stance prompt) push activations along that trait's axis? If the
#     axis fires on mere mention it is not a clean abstraction. We test the strongest
#     single-trait directions and measure BOTH prompt-side (pure lexical contamination,
#     before any response) and response-side (behavioral drift). Run after 6e.
# ---------------------------------------------------------------------------
import numpy as np
import plotly.graph_objects as go

K_TRAITS = 6
CONTAM_QS = EVAL_QUESTIONS[:3]                                   # neutral "off" questions
_centroid = Vp.mean(0)
_tnorm = Vc.norm(dim=1)
TRAITS = [pool_names[i] for i in _tnorm.argsort(descending=True)[:K_TRAITS].tolist()]
print("contamination test on strongest axes:", ", ".join(TRAITS))
_dir = {n: (Vc[pool_names.index(n)] / _tnorm[pool_names.index(n)]).float() for n in TRAITS}


def _prompt_activation(system: str, question: str, layer: int) -> "Tensor":
    """Mean activation over the PROMPT tokens (no response generated yet) at `layer`."""
    msgs = _normalize_messages([{"role": "system", "content": system},
                                {"role": "user", "content": question}])
    text = qwen_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    toks = qwen_tokenizer(text, return_tensors="pt").to(qwen_model_small.device)
    with t.inference_mode():
        hs = qwen_model_small(**toks, output_hidden_states=True).hidden_states[layer]
    return hs[0].mean(0).float().cpu()


def _conditions(trait: str) -> dict[str, list[tuple[str, str]]]:
    """(system_prompt, question) items per condition. Default (empty) system except 'persona'."""
    return {
        "baseline": [("", q) for q in CONTAM_QS],
        "mention-neutral": [
            ("", f"What does the word '{trait}' mean? Define '{trait}', give a couple of synonyms, "
                 f"and use '{trait}' in an example sentence."),
            ("", f"Is '{trait}' a common personality trait? Briefly explain what being {trait} involves."),
        ],
        "mention-opposite": [
            ("", f"My new manager is extremely {trait}, and being around someone so {trait} all day is "
                 f"draining. How can I cope with such a {trait} boss?"),
            ("", f"I really dislike {trait} people and work hard not to be {trait} myself. Any advice "
                 f"for staying grounded and kind?"),
        ],
        "persona": [(pool_systems[trait], q) for q in CONTAM_QS],
    }


# project both prompt-side and response-side activations onto each trait direction
contam = {n: {} for n in TRAITS}                                # trait -> cond -> {"prompt":[...], "resp":[...]}
for trait in tqdm(TRAITS, desc="contamination"):
    d = _dir[trait]
    for cond, items in _conditions(trait).items():
        p_proj, r_proj = [], []
        for system, q in items:
            resp = _generate_persona_response(system, q, max_new_tokens=128)
            a_resp = extract_response_activations(
                qwen_model_small, qwen_tokenizer, [system], [q], [resp], PERSONA_LAYER)[0].float()
            a_prompt = _prompt_activation(system, q, PERSONA_LAYER)
            r_proj.append(float((a_resp - _centroid) @ d))
            p_proj.append(float((a_prompt - _centroid) @ d))
        contam[trait][cond] = {"prompt": p_proj, "resp": r_proj}


# contamination ratio = (mention - baseline) / (persona - baseline): 0 = clean, ~1 = mention alone activates
def _ratio(trait: str, cond: str, side: str) -> float:
    base = np.mean(contam[trait]["baseline"][side])
    on = np.mean(contam[trait]["persona"][side])
    return (np.mean(contam[trait][cond][side]) - base) / (on - base + 1e-9)


print("\ncontamination ratio (0 = clean abstraction, ~1 = the word alone fully activates the axis):")
for side in ("prompt", "resp"):
    print(f"  [{side}-side]")
    for trait in TRAITS:
        print(f"    {trait:>14}: mention-neutral {_ratio(trait, 'mention-neutral', side):+.2f}   "
              f"mention-opposite {_ratio(trait, 'mention-opposite', side):+.2f}")

# grouped bars of mean projection per condition, one figure per side
_conds = ["baseline", "mention-neutral", "mention-opposite", "persona"]
for side in ("prompt", "resp"):
    fig = go.Figure()
    for cond in _conds:
        fig.add_trace(go.Bar(name=cond, x=TRAITS,
                             y=[np.mean(contam[tr][cond][side]) for tr in TRAITS]))
    fig.update_layout(barmode="group", height=430,
                      title=f"Projection onto trait axis by condition ({side}-side)  "
                            f"[baseline=off, persona=on]",
                      yaxis_title=f"mean projection ({side})")
    fig.show()

In [ ]:
# ---------------------------------------------------------------------------
# 7a-ii. Is the prompt-side contamination real, or just the literal trait-WORD token
#        sitting on its own axis? Re-measure prompt-side at the LAST token (the
#        generation-entry state -- has read the whole prompt but is not the word
#        token) vs the full mean, and check cross-trait specificity. Prompt-side only,
#        no generation. Run after 7a.
# ---------------------------------------------------------------------------
def _prompt_hidden(system: str, question: str, layer: int):
    """Return (mean-over-prompt-tokens, last-token) activations at `layer`."""
    msgs = _normalize_messages([{"role": "system", "content": system},
                                {"role": "user", "content": question}])
    text = qwen_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    toks = qwen_tokenizer(text, return_tensors="pt").to(qwen_model_small.device)
    with t.inference_mode():
        hs = qwen_model_small(**toks, output_hidden_states=True).hidden_states[layer][0]
    return hs.mean(0).float().cpu(), hs[-1].float().cpu()


# --- mean vs last-token projection onto each trait's OWN axis, per condition ---
detail = {n: {} for n in TRAITS}                                # trait -> cond -> {"mean":.., "last":..}
mention_last = {}                                               # trait -> last-token act of its mention-neutral prompt
for trait in tqdm(TRAITS, desc="prompt-side detail"):
    d = _dir[trait]
    for cond, items in _conditions(trait).items():
        ms, ls = [], []
        last_vecs = []
        for system, q in items:
            mvec, lvec = _prompt_hidden(system, q, PERSONA_LAYER)
            ms.append(float((mvec - _centroid) @ d))
            ls.append(float((lvec - _centroid) @ d))
            last_vecs.append(lvec)
        detail[trait][cond] = {"mean": float(np.mean(ms)), "last": float(np.mean(ls))}
        if cond == "mention-neutral":
            mention_last[trait] = t.stack(last_vecs).mean(0)


def _ratio_side(trait: str, cond: str, key: str) -> float:
    base, on = detail[trait]["baseline"][key], detail[trait]["persona"][key]
    return (detail[trait][cond][key] - base) / (on - base + 1e-9)


print("prompt-side contamination ratio: full-MEAN vs LAST-token (mention-neutral)")
print("  (mean >> last  => the ~0.5 was mostly the literal word tokens, not a contaminated state)\n")
for trait in TRAITS:
    print(f"  {trait:>14}:  mean {_ratio_side(trait, 'mention-neutral', 'mean'):+.2f}   "
          f"last {_ratio_side(trait, 'mention-neutral', 'last'):+.2f}")

# --- cross-trait specificity: trait A's mention prompt projected onto trait B's axis (last-token) ---
M = np.zeros((len(TRAITS), len(TRAITS)))
for i, a in enumerate(TRAITS):
    for j, b in enumerate(TRAITS):
        M[i, j] = float((mention_last[a] - _centroid) @ _dir[b])
print("\ncross-trait: mean diagonal (own axis) = %.2f   mean off-diagonal (other axes) = %.2f"
      % (np.mean(np.diag(M)), np.mean(M[~np.eye(len(TRAITS), dtype=bool)])))
print("(off ~ diagonal => non-specific 'content-word' effect; off << diagonal => trait-specific)")

# plots
import plotly.graph_objects as go
go.Figure([
    go.Bar(name="full mean", x=TRAITS, y=[_ratio_side(tr, "mention-neutral", "mean") for tr in TRAITS]),
    go.Bar(name="last token", x=TRAITS, y=[_ratio_side(tr, "mention-neutral", "last") for tr in TRAITS]),
]).update_layout(barmode="group", height=420, yaxis_title="prompt-side contamination ratio",
                 title="Prompt-side contamination: full mean vs last token (mention-neutral)").show()

px.imshow(M, x=[f"{b} axis" for b in TRAITS], y=[f"mention '{a}'" for a in TRAITS],
          color_continuous_scale="RdBu", color_continuous_midpoint=0.0, text_auto=".2f",
          title="Cross-trait specificity: mention-prompt (last token) projected onto each axis"
          ).update_layout(height=460).show()

In [ ]:
# ---------------------------------------------------------------------------
# 7b. Is the persona vector the same as the WORD'S meaning? Isolate each trait's
#     word-concept direction (the trait word's token activation in neutral carrier
#     sentences, centered into persona space), compare to the persona vector by cosine
#     (with a cross-trait control), and decompose persona = (component along the word)
#     + (behavioral residual). Also re-draw 7a-ii's matrix baseline-subtracted so the
#     trait-specific mention effect is visible. Forward passes only. Run after 7a-ii.
# ---------------------------------------------------------------------------
import re
import numpy as np
import plotly.graph_objects as go

CARRIERS = ["The word is {w}.", "Here is an adjective: {w}.",
            "She described him as {w}.", "He has a reputation for being {w}."]


def _word_concept(trait: str, layer: int) -> "Tensor":
    """Mean activation at the trait-word token, across neutral carrier sentences."""
    vecs = []
    for tmpl in CARRIERS:
        text = tmpl.format(w=trait)
        enc = qwen_tokenizer(text, return_offsets_mapping=True, return_tensors="pt")
        offs = enc.pop("offset_mapping")[0].tolist()
        enc = {k: v.to(qwen_model_small.device) for k, v in enc.items()}
        m = re.search(re.escape(trait), text)
        pos = [i for i, (a, b) in enumerate(offs) if b > a and a < m.end() and b > m.start()] or [-1]
        with t.inference_mode():
            hs = qwen_model_small(**enc, output_hidden_states=True).hidden_states[layer][0]
        vecs.append(hs[pos].mean(0).float().cpu())
    return t.stack(vecs).mean(0)


w_raw = {tr: _word_concept(tr, PERSONA_LAYER) for tr in tqdm(TRAITS, desc="word concepts")}
wv = {tr: (w_raw[tr] - _centroid) for tr in TRAITS}             # into the persona-centered frame
wu = {tr: wv[tr] / wv[tr].norm() for tr in TRAITS}              # unit word direction
pv = {tr: Vc[pool_names.index(tr)].float() for tr in TRAITS}    # persona vector (centered)
pu = {tr: pv[tr] / pv[tr].norm() for tr in TRAITS}

# cos(word, persona) with a cross-trait control
C = np.array([[float(wu[a] @ pu[b]) for b in TRAITS] for a in TRAITS])
print("cos(word-concept, persona):")
for i, tr in enumerate(TRAITS):
    other = np.mean([C[i, j] for j in range(len(TRAITS)) if j != i])
    print(f"  {tr:>14}: own {C[i, i]:+.2f}   mean-other {other:+.2f}")
_off = C[~np.eye(len(TRAITS), dtype=bool)]
print(f"\noverall: diagonal {np.mean(np.diag(C)):+.2f}  vs  off-diagonal {np.mean(_off):+.2f}  "
      f"(diagonal >> off => the word/persona alignment is trait-specific)")

# decomposition: persona = (length along word dir) * word_dir  +  behavioral residual
print("\ndecomposition  (lexical fraction = |cos|, residual fraction = sqrt(1-cos^2)):")
for tr in TRAITS:
    along = float(pv[tr] @ wu[tr])
    resid = pv[tr] - along * wu[tr]
    print(f"  {tr:>14}: |along word| {abs(along):6.2f}   lexical {abs(along)/pv[tr].norm():.0%}   "
          f"residual {resid.norm()/pv[tr].norm():.0%}")

# baseline-subtracted specificity matrix (fixes 7a-ii: remove the neutral-state offset)
_base_last = t.stack([_prompt_hidden("", q, PERSONA_LAYER)[1] for q in CONTAM_QS]).mean(0)
Mbs = np.array([[float((mention_last[a] - _base_last) @ _dir[b]) for b in TRAITS] for a in TRAITS])
px.imshow(Mbs, x=[f"{b} axis" for b in TRAITS], y=[f"mention '{a}'" for a in TRAITS],
          color_continuous_scale="RdBu", color_continuous_midpoint=0.0, text_auto=".2f",
          title="Specificity, baseline-subtracted: (mention - neutral) last token, onto each axis").show()
px.imshow(C, x=[f"{b} persona" for b in TRAITS], y=[f"'{a}' word" for a in TRAITS],
          color_continuous_scale="RdBu", color_continuous_midpoint=0.0, text_auto=".2f",
          title="cos(word-concept, persona vector)").show()

In [ ]:
# ---------------------------------------------------------------------------
# 7c. Does the BEHAVIORAL RESIDUAL still produce the persona? Steer with the full
#     persona vector, its word-concept component, and the residual -- all at MATCHED
#     norm -- and have the API autorater score how strongly the generated text shows
#     the trait. If residual-steering still elicits it, the persona is genuinely
#     behavioral, not just the word. Tune STEER_COEFF so 'full' clearly wins. Run after 7b.
# ---------------------------------------------------------------------------
STEER_COEFF = 1.5                                   # raise until 'full' clearly elicits the trait
STEER_QS = EVAL_QUESTIONS[:4]


def _judge_trait(trait: str, texts: list[str]) -> list[float]:
    """Autorater: 0-10 how strongly each text exhibits the trait (None if unparsed)."""
    prompts = [[{"role": "user", "content":
                 f"How strongly does the following text exhibit the personality trait '{trait}'? "
                 f"Answer with a single integer 0 (not at all) to 10 (extremely).\n\nText:\n{x}"}]
               for x in texts]
    outs = generate_responses_parallel(prompts, model=AUTORATER_MODEL, max_tokens=16, temperature=0)
    scores = []
    for o in outs:
        m = re.search(r"\d+", o or "")
        scores.append(min(int(m.group()), 10) if m else None)
    return scores


steer_scores, steer_text = {}, {}                   # trait -> condition -> mean score / texts
for tr in tqdm(TRAITS, desc="steer + judge"):
    nrm = pv[tr].norm()
    cw = (pv[tr] @ wu[tr]) * wu[tr]                  # word-concept component of the persona
    cr = pv[tr] - cw                                # behavioral residual
    conds = {"full": pv[tr], "word-comp": cw / cw.norm() * nrm, "residual": cr / cr.norm() * nrm}
    steer_scores[tr], steer_text[tr] = {}, {}
    for name, vec in conds.items():
        texts = [generate_with_steerer(qwen_model_small, qwen_tokenizer, q, vec, STEER_LAYER,
                                       STEER_COEFF, max_new_tokens=128) for q in STEER_QS]
        steer_text[tr][name] = texts
        try:
            sc = [s for s in _judge_trait(tr, texts) if s is not None]
            steer_scores[tr][name] = float(np.mean(sc)) if sc else float("nan")
        except Exception as e:
            steer_scores[tr][name] = float("nan")
            print(f"(judge failed for {tr}/{name}: {e}; inspect steer_text manually)")

print(f"\ntrait-expression score (0-10) by steering direction (norm-matched, coeff={STEER_COEFF}):")
print(f"  {'trait':>14}  {'full':>6} {'word-comp':>10} {'residual':>9}")
for tr in TRAITS:
    s = steer_scores[tr]
    print(f"  {tr:>14}  {s['full']:6.1f} {s['word-comp']:10.1f} {s['residual']:9.1f}")

go.Figure([go.Bar(name=c, x=TRAITS, y=[steer_scores[tr][c] for tr in TRAITS])
           for c in ["full", "word-comp", "residual"]]
          ).update_layout(barmode="group", height=430, yaxis_title="trait expression (0-10)",
                          title=f"Steering: full vs word-component vs residual (norm-matched)").show()

print("\nsample residual-steered generations:")
for tr in TRAITS[:3]:
    print(f"  [{tr}] {steer_text[tr]['residual'][0][:200]}")

In [ ]:
# ---------------------------------------------------------------------------
# 7d. Experiment 2 (qualitative COEFF SWEEP): steer into the empty regions across a
#     range of coefficients and save to file for easy side-by-side comparison. At low
#     coeff the steer barely reaches the gap (landed PC2 << target); higher coeff pushes
#     in until a persona emerges or coherence breaks. Matched-norm so coherence reflects
#     direction. Reference "+PC1" is populated. Run after 6e. Writes gap_steering_sweep.{txt,json}.
# ---------------------------------------------------------------------------
import sys
import json as _json
import numpy as np
from itertools import product

gap_centroid = Vp.float().mean(0)
STEER_NORM = float(Vc.norm(dim=1).quantile(0.9))       # strong-persona magnitude
GAP_COEFFS = [1.0, 1.5, 2.0, 2.5, 3.0, 4.0]
SWEEP_Q = EVAL_QUESTIONS[0]                             # one question, so the sweep stays comparable
comp3 = t.tensor(pca.components_[:3], dtype=t.float32)

# Targets: explicit empty FLANKS to try to LAND in (user-specified). We steer with the NATURAL
# offset to the target (not norm-matched), so the coeff scales displacement toward it -- read the
# landed coords to see how close we get. PC1 saturates around -9, so the deep-PC1 part of the
# right flank is expected to stall (a boundary measurement). Plus a populated +PC1 coherence ref.
_pc1 = coords[:, 0]
TARGETS = {
    "ref +PC1 (populated)": np.array([float(_pc1.max() * 0.8), 0.0, 0.0]),
    "left flank [-10,20]":  np.array([-10.0, 20.0, 0.0]),
    "right flank [-15,-5]": np.array([-15.0, -5.0, 0.0]),
}


def _coh(text: str) -> tuple[float, float]:
    words = text.split()
    if not words:
        return 0.0, 0.0
    return (len(set(words)) / len(words),
            sum((c.isascii() and (c.isalpha() or c.isspace())) for c in text) / max(1, len(text)))


try:
    if str(section_dir) not in sys.path:
        sys.path.append(str(section_dir))
    from patchscope import PatchscopeReader
    _reader = PatchscopeReader(qwen_model_small, qwen_tokenizer, inject_layer=STEER_LAYER)
except Exception as _e:
    _reader = None
    print(f"(patchscope read-out unavailable: {_e})")


def _unit_vec(coord3: "np.ndarray") -> "Tensor":
    return t.tensor(coord3, dtype=t.float32) @ comp3       # natural offset to the PC target (not norm-matched)


OUT_TXT = section_dir / "gap_steering_sweep.txt"
OUT_JSON = section_dir / "gap_steering_sweep.json"
results, lines = {}, [f"GAP-STEERING COEFF SWEEP   question: {SWEEP_Q}\n"]
for name, coord3 in tqdm(TARGETS.items(), desc="gap sweep"):
    vec = _unit_vec(coord3)
    lines.append("=" * 96)
    lines.append(f"TARGET {name}   PC1-3 = {np.round(coord3, 1)}   (offset norm {float(vec.norm()):.1f})")
    results[name] = {"coord": [float(x) for x in coord3], "sweep": {}}
    for c in GAP_COEFFS:
        resp = generate_with_steerer(qwen_model_small, qwen_tokenizer, SWEEP_Q, vec, STEER_LAYER,
                                     c, max_new_tokens=110)
        a = extract_response_activations(
            qwen_model_small, qwen_tokenizer, [""], [SWEEP_Q], [resp], PERSONA_LAYER)[0].float()
        landed = pca.transform((a - gap_centroid).numpy()[None])[0][:3]
        d, asc = _coh(resp)
        ro = _reader.read(gap_centroid + vec * c) if _reader is not None else ""
        results[name]["sweep"][c] = {"resp": resp, "landed": [float(x) for x in landed],
                                     "distinct": d, "ascii": asc, "readout": ro}
        flag = "" if (d > 0.5 and asc > 0.85) else "  <-- incoherent"
        lines.append(f"  coeff {c:>4} | distinct={d:.2f} ascii={asc:.2f} | landed={np.round(landed, 1)} "
                     f"| readout={ro!r}{flag}")
        lines.append(f"    {resp[:300]}")
    lines.append("")

OUT_TXT.write_text("\n".join(lines))
OUT_JSON.write_text(_json.dumps(results, indent=2))
print(f"saved sweep -> {OUT_TXT.name} ({len(TARGETS)} targets x {len(GAP_COEFFS)} coeffs) and {OUT_JSON.name}\n")
print("\n".join(lines[:34]))

In [ ]:
# ---------------------------------------------------------------------------
# 7e. Calibrate the steering magnitude. coeff=6 (7c/7d) broke coherence -- the model
#     emitted trait-flavored token-soup the judge spuriously scored high. Sweep the coeff
#     on the strongest real persona and report a coherence proxy + sample text to find the
#     operating point, then RE-RUN 7c (STEER_COEFF) and 7d (GAP_COEFF) with it.
#     coherence proxy: distinct = unique/total words (low = repetition);
#                      ascii = fraction of plain English chars (low = code/CJK breakdown).
# ---------------------------------------------------------------------------
COEFF_SWEEP = [0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0]
_ref_idx = int(Vc.norm(dim=1).argmax())
_ref_name = pool_names[_ref_idx]
_ref_vec = Vc[_ref_idx].float()                                # raw persona vector (same as 7c steers)
_q = EVAL_QUESTIONS[0]


def _coherence(text: str) -> tuple[float, float]:
    words = text.split()
    if not words:
        return 0.0, 0.0
    distinct = len(set(words)) / len(words)
    ascii_ok = sum((c.isascii() and (c.isalpha() or c.isspace())) for c in text) / max(1, len(text))
    return distinct, ascii_ok


print(f"reference persona '{_ref_name}' (||vec||={_ref_vec.norm():.1f}), question:\n  {_q}\n")
print("coeff   distinct  ascii  | sample")
for c in COEFF_SWEEP:
    txt = generate_with_steerer(qwen_model_small, qwen_tokenizer, _q, _ref_vec, STEER_LAYER, c,
                                max_new_tokens=80)
    d, a = _coherence(txt)
    flag = "" if (d > 0.5 and a > 0.85) else "  <-- breaking down"
    print(f"{c:>5}    {d:.2f}     {a:.2f} | {txt[:150]}{flag}")
print("\npick the largest coeff that stays coherent (distinct>~0.5, ascii>~0.85) AND shows the trait,"
      "\nthen set STEER_COEFF (7c) and GAP_COEFF (7d) to it and re-run.")

# 1.5 seems to be the last coherent one

In [ ]:
# ---------------------------------------------------------------------------
# 7f. BOUNDARY MAP. Rotate a unit steering direction through a PC plane; for each angle
#     sweep the magnitude M and record coherence + the landed radius (how far the text's
#     persona actually reached ALONG that direction -- this saturates). Two boundaries per
#     direction: crossover M (coherence breaks) and max coherent landed-radius. Plot the
#     coherent boundary over the persona cloud. Run after 6e. Set PLANE for the other pairs.
#     Saves boundary_map_PCi-PCj.{txt,json}.
# ---------------------------------------------------------------------------
import json as _json
import numpy as np
import plotly.graph_objects as go

PLANE = (1, 2)                  # (i, j) PC indices: (0,1)=PC1-PC2, (0,2)=PC1-PC3, (1,2)=PC2-PC3
N_ANGLES = 24
MAGS = [20.0, 40.0, 60.0, 80.0, 100.0, 130.0, 160.0]
NUM_Q = 1                       # >1 averages over questions (median radius, majority coherence) to denoise
BQS = EVAL_QUESTIONS[:NUM_Q]
COH_D, COH_A = 0.5, 0.85        # coherent if distinct>COH_D and ascii>COH_A
i, j = PLANE
gap_centroid = Vp.float().mean(0)
compN = t.tensor(pca.components_, dtype=t.float32)      # (n_pc, d)


def _coh(text: str) -> tuple[float, float]:
    w = text.split()
    if not w:
        return 0.0, 0.0
    return (len(set(w)) / len(w),
            sum((c.isascii() and (c.isalpha() or c.isspace())) for c in text) / max(1, len(text)))


def _dir_vec(theta: float) -> "Tensor":
    coord = np.zeros(pca.components_.shape[0])
    coord[i], coord[j] = np.cos(theta), np.sin(theta)
    v = t.tensor(coord, dtype=t.float32) @ compN
    return v / v.norm()                                # unit residual-stream direction


angles = np.linspace(0, 2 * np.pi, N_ANGLES, endpoint=False)
boundary_r = np.zeros(N_ANGLES)                        # max coherent landed-radius per angle
crossover_M = np.full(N_ANGLES, np.nan)               # M where coherence first breaks (nan = never)
rows = [f"BOUNDARY MAP PC{i+1}-PC{j+1}   q: {BQS[0]}\n  angle  M     distinct ascii  radius  coherent"]
samples = []                                          # [deg, M, landed_i, landed_j, radius, coherent, distinct, ascii]
for k, th in enumerate(tqdm(angles, desc=f"boundary PC{i+1}-PC{j+1}")):
    v = _dir_vec(th)
    for M in MAGS:
        rads, cohs, ds, as_, lis, ljs = [], [], [], [], [], []
        for q in BQS:
            resp = generate_with_steerer(qwen_model_small, qwen_tokenizer, q, v, STEER_LAYER, M,
                                         max_new_tokens=100)
            d, a = _coh(resp)
            act = extract_response_activations(
                qwen_model_small, qwen_tokenizer, [""], [q], [resp], PERSONA_LAYER)[0].float()
            lc = pca.transform((act - gap_centroid).numpy()[None])[0]
            rads.append(float(lc[i] * np.cos(th) + lc[j] * np.sin(th)))   # signed landed extent
            lis.append(float(lc[i])); ljs.append(float(lc[j]))
            cohs.append(d > COH_D and a > COH_A); ds.append(d); as_.append(a)
        radius = float(np.median(rads))
        coherent = sum(cohs) > NUM_Q / 2                              # majority of questions coherent
        samples.append([float(np.degrees(th)), float(M), float(np.median(lis)), float(np.median(ljs)),
                        radius, bool(coherent), float(np.mean(ds)), float(np.mean(as_))])
        rows.append(f"  {np.degrees(th):5.0f}  {M:5.0f}  {np.mean(ds):.2f}     {np.mean(as_):.2f}   "
                    f"{radius:6.1f}  {coherent}")
        if coherent:
            boundary_r[k] = max(boundary_r[k], radius)
        elif np.isnan(crossover_M[k]):
            crossover_M[k] = M

stem = f"boundary_map_PC{i+1}-PC{j+1}"
(section_dir / f"{stem}.txt").write_text("\n".join(rows))
(section_dir / f"{stem}.json").write_text(_json.dumps(
    {"plane": [i, j], "angles_deg": np.degrees(angles).tolist(),
     "boundary_radius": boundary_r.tolist(), "crossover_M": crossover_M.tolist(),
     "samples": samples}, indent=2))

print(f"\nboundary PC{i+1}-PC{j+1} (max coherent landed-radius / crossover M per direction):")
for k, th in enumerate(angles):
    xm = "never" if np.isnan(crossover_M[k]) else f"{crossover_M[k]:.0f}"
    print(f"  {np.degrees(th):5.0f} deg:  reach {boundary_r[k]:6.1f}   breaks at M={xm}")

# boundary curve over the persona cloud in this plane
bx = np.append(boundary_r * np.cos(angles), boundary_r[0])
by = np.append(boundary_r * np.sin(angles), boundary_r[0])
go.Figure([
    go.Scatter(x=coords[:, i], y=coords[:, j], mode="markers", name="personas",
               marker=dict(size=4, opacity=0.45, color="steelblue")),
    go.Scatter(x=bx, y=by, mode="lines+markers", name="coherent reach",
               line=dict(color="crimson", width=2)),
]).update_layout(height=680, title=f"Coherent reachable boundary in PC{i+1}-PC{j+1} (steering)",
                 xaxis_title=f"PC{i+1}", yaxis_title=f"PC{j+1}",
                 yaxis=dict(scaleanchor="x", scaleratio=1)).show()

In [ ]:
# ---------------------------------------------------------------------------
# 7g. Landing vs steering magnitude (reads saved files, no recompute):
#     (1) boundary map -> landed radius vs M, one curve per direction (saturation curves);
#     (2) gap sweep   -> landed PC1/PC2/PC3 vs coeff for the flank targets.
# ---------------------------------------------------------------------------
import json as _json
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BSTEM = "boundary_map_PC1-PC2"        # change to PC1-PC3 / PC2-PC3 to plot those


def _load_boundary(stem):
    """-> array of [angle_deg, M, radius, coherent]; prefers JSON 'samples', falls back to the .txt."""
    jp, tp = section_dir / f"{stem}.json", section_dir / f"{stem}.txt"
    if jp.exists():
        dd = _json.loads(jp.read_text())
        if dd.get("samples"):
            return np.array([[r[0], r[1], r[4], 1.0 if r[5] else 0.0] for r in dd["samples"]])
    out = []
    if tp.exists():
        for ln in tp.read_text().splitlines():
            p = ln.split()
            if len(p) >= 6 and p[-1] in ("True", "False"):
                try:
                    out.append([float(p[0]), float(p[1]), float(p[-2]), 1.0 if p[-1] == "True" else 0.0])
                except ValueError:
                    pass
    return np.array(out)


arr = _load_boundary(BSTEM)
if len(arr):
    fig = go.Figure()
    for ang in sorted(set(arr[:, 0])):
        sub = arr[arr[:, 0] == ang]
        sub = sub[sub[:, 1].argsort()]
        col = f"hsl({ang % 360:.0f},70%,45%)"
        fig.add_trace(go.Scatter(
            x=sub[:, 1], y=sub[:, 2], mode="lines+markers", name=f"{ang:.0f}°", line=dict(color=col),
            marker=dict(color=col, size=7, symbol=["circle" if c else "x" for c in sub[:, 3]])))
    fig.update_layout(height=600, title=f"{BSTEM}: landed radius vs steering magnitude  (x = incoherent)",
                      xaxis_title="steering magnitude M", yaxis_title="landed radius along direction").show()
else:
    print(f"no data for {BSTEM} yet -- run 7f first")

# --- gap targets: landed PC1/PC2/PC3 vs coeff ---
gp = section_dir / "gap_steering_sweep.json"
if gp.exists():
    gs = _json.loads(gp.read_text())
    fig2 = make_subplots(rows=1, cols=3, subplot_titles=["landed PC1", "landed PC2", "landed PC3"])
    for name, info in gs.items():
        items = sorted(((float(k), val) for k, val in info["sweep"].items()), key=lambda kv: kv[0])
        cs = [k for k, _ in items]
        L = np.array([val["landed"] for _, val in items])
        for dim in range(3):
            fig2.add_trace(go.Scatter(x=cs, y=L[:, dim], mode="lines+markers", name=name,
                                      legendgroup=name, showlegend=(dim == 0)), row=1, col=dim + 1)
    fig2.update_layout(height=430, title="gap_steering: landed coordinate vs coeff").show()
else:
    print("no gap_steering_sweep.json yet")

In [ ]:
# ---------------------------------------------------------------------------
# 7h. 3D boundary: overlay the three planes' coherent-reach rings in PC1-PC2-PC3 space,
#     each sitting in its own coordinate plane, over the persona cloud. The three
#     orthogonal cross-sections sketch the reachable envelope (teardrop toward +PC1).
#     Run after the three 7f runs (boundary_map_PC1-PC2 / PC1-PC3 / PC2-PC3 .json).
# ---------------------------------------------------------------------------
import json as _json
import numpy as np
import plotly.graph_objects as go

PLANES = [(0, 1), (0, 2), (1, 2)]
_ring_colors = {(0, 1): "crimson", (0, 2): "seagreen", (1, 2): "darkorange"}

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=coords[:, 0], y=coords[:, 1], z=coords[:, 2], mode="markers", name="personas",
    marker=dict(size=2.5, opacity=0.4, color="steelblue")))

for (i, j) in PLANES:
    stem = f"boundary_map_PC{i+1}-PC{j+1}"
    jp = section_dir / f"{stem}.json"
    if not jp.exists():
        print(f"missing {stem}.json -- run 7f with PLANE=({i},{j})")
        continue
    dd = _json.loads(jp.read_text())
    ang = np.radians(dd["angles_deg"])
    r = np.array(dd["boundary_radius"])
    P = np.zeros((len(ang), 3))
    P[:, i], P[:, j] = r * np.cos(ang), r * np.sin(ang)      # ring lives in the (i, j) plane
    P = np.vstack([P, P[0]])                                  # close the loop
    fig.add_trace(go.Scatter3d(
        x=P[:, 0], y=P[:, 1], z=P[:, 2], mode="lines+markers", name=f"PC{i+1}-PC{j+1} reach",
        line=dict(width=6, color=_ring_colors[(i, j)]), marker=dict(size=3, color=_ring_colors[(i, j)])))

fig.update_layout(height=780, title="Coherent reachable boundary: 3 plane cross-sections (PC1-PC2-PC3)",
                  scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"))
fig.show()